# 02 — RAG Walkthrough

Pipeline RAG passo a passo: carregamento → chunking → embedding → armazenamento → recuperação → geração.

## Passo 1: Carregar Documento

In [1]:
from modules.rag.document_loader import load_document
doc = load_document('../modules/rag/data/sample_docs/genai_concepts.md')
print(f'Documento: {doc["metadata"]["source"]}')
print(f'Tamanho: {len(doc["text"])} caracteres')
print('\nPrimeiros 500 chars:')
print(doc['text'][:500])

[00:51:41] INFO     Carregando: genai_concepts.md                                             ]8;id=9281142;file://C:\Desenvolvimento\IA\Aprendizado\ProjetoInicial\modules\rag\document_loader.py\document_loader.py]8;;\:]8;id=9281143;file://C:\Desenvolvimento\IA\Aprendizado\ProjetoInicial\modules\rag\document_loader.py#57\57]8;;\

Documento: genai_concepts.md
Tamanho: 3592 caracteres

Primeiros 500 chars:
# Conceitos de IA Generativa

## Large Language Models (LLMs)

Large Language Models (LLMs) são modelos de aprendizado profundo treinados em grandes quantidades de texto. Eles aprendem a prever o próximo token em uma sequência, o que lhes dá a capacidade emergente de gerar texto coerente, responder perguntas, traduzir idiomas e realizar raciocínio complexo.

Os LLMs modernos são baseados na arquitetura Transformer, introduzida no paper "Attention Is All You Need" (Vaswani et al., 2017). Os compo


## Passo 2: Chunking — 3 estratégias

In [2]:
from modules.rag.chunker import fixed_size_chunk, sentence_chunk, recursive_chunk
text = doc['text']
fixed = fixed_size_chunk(text, size=400, overlap=50)
sentence = sentence_chunk(text, max_sentences=4)
recursive = recursive_chunk(text, max_size=400, overlap=50)
print(f'Fixed-size: {len(fixed)} chunks')
print(f'Sentence:   {len(sentence)} chunks')
print(f'Recursive:  {len(recursive)} chunks')
print('\nPrimeiro chunk (recursive):')
print(recursive[0])

Fixed-size: 11 chunks
Sentence:   6 chunks
Recursive:  12 chunks

Primeiro chunk (recursive):
# Conceitos de IA Generativa

## Large Language Models (LLMs)

Large Language Models (LLMs) são modelos de aprendizado profundo treinados em grandes quantidades de texto. Eles aprendem a prever o próximo token em uma sequência, o que lhes dá a capacidade emergente de gerar texto coerente, responder perguntas, traduzir idiomas e realizar raciocínio complexo.


## Passo 3: Embeddings

In [3]:
from modules.embeddings.local_embedder import LocalEmbedder
emb = LocalEmbedder()
vectors = emb.embed_batch(recursive[:5])
print(f'Dimensão: {len(vectors[0])}')
print(f'Primeiro vetor (5 primeiros valores): {vectors[0][:5]}')

[00:52:25] INFO     Carregando modelo de embedding: all-MiniLM-L6-v2                           ]8;id=9281150;file://C:\Desenvolvimento\IA\Aprendizado\ProjetoInicial\modules\embeddings\local_embedder.py\local_embedder.py]8;;\:]8;id=9281151;file://C:\Desenvolvimento\IA\Aprendizado\ProjetoInicial\modules\embeddings\local_embedder.py#17\17]8;;\

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Desenvolvimento\IA\Aprendizado\ProjetoInicial\modules\embeddings\local_embedder.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dim: int = self._model.get_sentence_embedding_dimension()


[00:52:29] INFO     Modelo carregado — dimensão: 384                                           ]8;id=9281157;file://C:\Desenvolvimento\IA\Aprendizado\ProjetoInicial\modules\embeddings\local_embedder.py\local_embedder.py]8;;\:]8;id=9281158;file://C:\Desenvolvimento\IA\Aprendizado\ProjetoInicial\modules\embeddings\local_embedder.py#20\20]8;;\

Dimensão: 384
Primeiro vetor (5 primeiros valores): [0.08818210661411285, -0.10372183471918106, 0.03112022951245308, -0.006461276672780514, 0.005291201174259186]


## Passo 4: Armazenar no ChromaDB

In [4]:
from modules.vector_store.chroma_store import ChromaVectorStore
store = ChromaVectorStore('notebook_rag', persist_dir='../modules/vector_store/chroma_db')
store.reset()
all_chunks = recursive
all_vecs = emb.embed_batch(all_chunks)
metadatas = [{'source': 'genai_concepts.md', 'chunk_idx': i} for i in range(len(all_chunks))]
store.add(all_chunks, all_vecs, metadatas)
print(f'Armazenados: {store.count()} chunks')

[00:52:35] INFO     ChromaDB collection 'notebook_rag' carregada (12 docs)                       ]8;id=9281165;file://C:\Desenvolvimento\IA\Aprendizado\ProjetoInicial\modules\vector_store\chroma_store.py\chroma_store.py]8;;\:]8;id=9281166;file://C:\Desenvolvimento\IA\Aprendizado\ProjetoInicial\modules\vector_store\chroma_store.py#29\29]8;;\

           WARNING  Collection 'notebook_rag' resetada.                                          ]8;id=9281172;file://C:\Desenvolvimento\IA\Aprendizado\ProjetoInicial\modules\vector_store\chroma_store.py\chroma_store.py]8;;\:]8;id=9281173;file://C:\Desenvolvimento\IA\Aprendizado\ProjetoInicial\modules\vector_store\chroma_store.py#92\92]8;;\

           INFO     Adicionados 12 documentos. Total: 12                                         ]8;id=9281179;file://C:\Desenvolvimento\IA\Aprendizado\ProjetoInicial\modules\vector_store\chroma_store.py\chroma_store.py]8;;\:]8;id=9281180;file://C:\Desenvolvimento\IA\Aprendizado\ProjetoInicial\modules\vector_store\chroma_store.py#46\46]8;;\

Armazenados: 12 chunks


## Passo 5: Recuperar e Gerar

In [5]:
from shared.llm_client import build_client
query = 'Quais são as vantagens do RAG sobre o fine-tuning?'
q_vec = emb.embed_one(query)
results = store.query(q_vec, n_results=3)
print(f'Top 3 chunks para: "{query}"\n')
for r in results:
    print(f'[{r["score"]:.3f}] {r["text"][:150]}...')
    print()
client = build_client()
context = '\n\n'.join(r['text'] for r in results)
prompt = f'Com base nos documentos:\n\n{context}\n\nPergunta: {query}'
response = client.complete([{'role': 'user', 'content': prompt}], max_tokens=512)
print('RESPOSTA RAG:')
print(response)

Top 3 chunks para: "Quais são as vantagens do RAG sobre o fine-tuning?"

[0.595] ### Vantagens do RAG sobre Fine-tuning

- **Atualização de conhecimento:** Novos documentos podem ser adicionados sem retreinar o modelo
- **Rastreabi...

[0.454] **Fase de Consulta:**
1. A pergunta do usuário é convertida em embedding
2. Os chunks mais similares são recuperados do vector store
3. Os chunks recu...

[0.446] ### Pipeline do RAG

**Fase de Ingestão:**
1. Documentos são carregados de diversas fontes (PDFs, bancos de dados, websites)
2. Os documentos são divi...

RESPOSTA RAG:
## Vantagens do RAG sobre o Fine-tuning

Com base nos documentos fornecidos, o RAG apresenta **quatro vantagens principais** em relação ao fine-tuning:

### 1. 🔄 Atualização de Conhecimento
Novos documentos podem ser **adicionados sem necessidade de retreinar** o modelo, tornando a atualização muito mais ágil.

### 2. 🔍 Rastreabilidade
As **fontes podem ser citadas nas respostas**, permitindo maior transparência e verifi

## Exercícios

1. Compare os 3 tipos de chunking — qual recupera chunks mais relevantes para a mesma query?
2. Tente a técnica HyDE: gere um documento hipotético e use-o como query
3. Adicione metadados de data aos chunks e filtre por `year >= 2023`